# GDS Lab: PageRank and Betweenness on the Airport Route Network

This notebook runs two centrality algorithms on the flight network — PageRank and
Betweenness Centrality — then pulls the scores back into Databricks to answer a
cross-database question: **do the most central airports in the route network also
generate the most maintenance-caused flight delays?**

## What You'll Learn
- How to build a weighted airport co-occurrence graph with Cypher aggregation
- The difference between PageRank (influence) and Betweenness (connector) centrality
- How to write GDS scores back to Neo4j node properties
- How to join graph-computed scores with Databricks flight delay data

## Why These Two Algorithms Together
PageRank and Betweenness often disagree. A high-PageRank airport is a traffic hub.
A high-Betweenness airport is a structural connector — removing it would split the
network. They can be different airports, and that difference is operationally
meaningful for disruption risk planning.

## Prerequisites
- Neo4j Aura credentials (AuraDB Professional or higher)
- Full dataset loaded via `02_load_neo4j_full.ipynb`

## Instructions
1. Enter credentials in Section 1
2. Run all cells in order

## Section 1: Configuration

In [ ]:
%pip install neo4j

In [ ]:
# ============================================
# CONFIGURATION
# ============================================

NEO4J_URI      = ""      # e.g., "neo4j+s://xxxxxxxx.databases.neo4j.io"
NEO4J_USERNAME = "neo4j"
NEO4J_PASSWORD = ""

if not NEO4J_URI or not NEO4J_PASSWORD:
    print("WARNING: Please enter your Neo4j credentials above before running!")
else:
    print(f"Configuration ready — {NEO4J_URI}")

In [ ]:
from neo4j import GraphDatabase

driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USERNAME, NEO4J_PASSWORD))

def run_query(cypher, params=None):
    """Execute a Cypher query and return results as a list of dicts."""
    with driver.session() as session:
        return [dict(r) for r in session.run(cypher, params or {})]

spark.conf.set("neo4j.url", NEO4J_URI)
spark.conf.set("neo4j.authentication.basic.username", NEO4J_USERNAME)
spark.conf.set("neo4j.authentication.basic.password", NEO4J_PASSWORD)
spark.conf.set("neo4j.database", "neo4j")

def spark_query(cypher):
    """Read Neo4j query results as a Spark DataFrame."""
    return (spark.read
        .format("org.neo4j.spark.DataSource")
        .option("query", cypher)
        .load())

result = run_query("RETURN 'Connected to Neo4j' AS status")
print(result[0]["status"])

## Section 2: Explore the Flight Network

Understand the shape of the airport network before running algorithms.

In [ ]:
# Airport traffic overview
airport_traffic = run_query("""
    MATCH (f:Flight)-[:DEPARTS_FROM]->(ap:Airport)
    WITH ap, count(f) AS departures
    OPTIONAL MATCH (f2:Flight)-[:ARRIVES_AT]->(ap)
    RETURN ap.iata     AS iata,
           ap.city     AS city,
           departures,
           count(f2)   AS arrivals
    ORDER BY departures DESC
""")

print(f"{'IATA':<6} {'City':<28} {'Departures':>12} {'Arrivals':>10}")
print("-" * 60)
for r in airport_traffic:
    print(f"{r['iata']:<6} {r['city']:<28} {r['departures']:>12} {r['arrivals']:>10}")

In [ ]:
# Top routes by frequency
routes = run_query("""
    MATCH (f:Flight)-[:DEPARTS_FROM]->(dep:Airport)
    MATCH (f)-[:ARRIVES_AT]->(arr:Airport)
    RETURN dep.iata AS origin,
           arr.iata AS destination,
           count(f) AS flight_count
    ORDER BY flight_count DESC
    LIMIT 15
""")

print(f"{'Route':<14} {'Flights':>8}")
print("-" * 25)
for r in routes:
    print(f"{r['origin']} → {r['destination']:<8} {r['flight_count']:>8}")

## Section 3: Build the Airport Route Projection

There are no direct Airport-to-Airport relationships in the graph — flights connect
them through intermediate Flight nodes. We use **Cypher aggregation projection** to
build an Airport-to-Airport virtual graph weighted by the number of flights on each
route.

A route with 10 flights gets edge weight 10, making high-frequency routes more
influential in the PageRank calculation.

In [ ]:
run_query("CALL gds.graph.drop('airport-routes', false) YIELD graphName")
print("Cleared previous projection.")

In [ ]:
result = run_query("""
    MATCH (dep:Airport)<-[:DEPARTS_FROM]-(f:Flight)-[:ARRIVES_AT]->(arr:Airport)
    WITH dep, arr, count(f) AS flight_count
    RETURN gds.graph.project(
        'airport-routes',
        dep,
        arr,
        {
            sourceNodeLabels: ['Airport'],
            targetNodeLabels: ['Airport'],
            relationshipType: 'FLIES_TO',
            relationshipProperties: {weight: flight_count}
        },
        {undirectedRelationshipTypes: ['FLIES_TO']}
    )
""")

proj = result[0]
print(f"Projection created:")
print(f"  Airports: {proj['nodeCount']}")
print(f"  Routes:   {proj['relationshipCount']}")

## Section 4: PageRank — Which Airports Are Most Influential?

PageRank scores an airport by both the volume of connections and the importance
of the airports it connects to. Using `relationshipWeightProperty: 'weight'` means
high-frequency routes contribute more influence.

In [ ]:
pagerank = run_query("""
    CALL gds.pageRank.stream('airport-routes', {
        maxIterations: 20,
        dampingFactor: 0.85,
        relationshipWeightProperty: 'weight'
    })
    YIELD nodeId, score
    RETURN gds.util.asNode(nodeId).iata  AS iata,
           gds.util.asNode(nodeId).city  AS city,
           round(score, 4)               AS pagerank_score
    ORDER BY pagerank_score DESC
""")

print(f"{'Rank':<6} {'IATA':<6} {'City':<28} {'PageRank':>10}")
print("-" * 55)
for i, r in enumerate(pagerank, 1):
    print(f"{i:<6} {r['iata']:<6} {r['city']:<28} {r['pagerank_score']:>10}")

## Section 5: Betweenness Centrality — Which Airports Are Critical Connectors?

Betweenness measures how often an airport lies on the shortest path between any
two other airports. A high-betweenness airport is a structural connector — removing
it would fragment the network most severely.

Compare the ranking with PageRank. Airports that rank high on both are simultaneously
traffic hubs and critical connectors — the highest disruption risk.

In [ ]:
betweenness = run_query("""
    CALL gds.betweenness.stream('airport-routes')
    YIELD nodeId, score
    RETURN gds.util.asNode(nodeId).iata  AS iata,
           gds.util.asNode(nodeId).city  AS city,
           round(score, 2)               AS betweenness_score
    ORDER BY betweenness_score DESC
""")

print(f"{'Rank':<6} {'IATA':<6} {'City':<28} {'Betweenness':>12}")
print("-" * 56)
for i, r in enumerate(betweenness, 1):
    print(f"{i:<6} {r['iata']:<6} {r['city']:<28} {r['betweenness_score']:>12}")

In [ ]:
# Side-by-side rank comparison — do PageRank and Betweenness agree?
pr_rank = {r["iata"]: i for i, r in enumerate(pagerank, 1)}
bt_rank = {r["iata"]: i for i, r in enumerate(betweenness, 1)}
pr_score = {r["iata"]: r["pagerank_score"] for r in pagerank}

all_iatas = sorted(pr_rank.keys())

print(f"{'IATA':<6} {'City':<28} {'PR Rank':>8} {'BT Rank':>8} {'Delta':>8}")
print("-" * 62)
city_map = {r["iata"]: r["city"] for r in pagerank}
for iata in sorted(all_iatas, key=lambda x: pr_rank.get(x, 99)):
    pr = pr_rank.get(iata, "–")
    bt = bt_rank.get(iata, "–")
    delta = abs(pr - bt) if isinstance(pr, int) and isinstance(bt, int) else "–"
    city = city_map.get(iata, "")
    print(f"{iata:<6} {city:<28} {str(pr):>8} {str(bt):>8} {str(delta):>8}")

## Section 6: Write Centrality Scores Back to Neo4j

In [ ]:
# Write PageRank scores
result = run_query("""
    CALL gds.pageRank.write('airport-routes', {
        writeProperty: 'pagerank_score',
        maxIterations: 20,
        relationshipWeightProperty: 'weight'
    })
    YIELD nodePropertiesWritten
""")
print(f"PageRank written to {result[0]['nodePropertiesWritten']} Airport nodes")

# Write Betweenness scores
result = run_query("""
    CALL gds.betweenness.write('airport-routes', {
        writeProperty: 'betweenness_score'
    })
    YIELD nodePropertiesWritten
""")
print(f"Betweenness written to {result[0]['nodePropertiesWritten']} Airport nodes")

## Section 7: Correlate Centrality with Maintenance Delays in Databricks

Pull centrality scores into Databricks and join with flight delay data. The key
question: **do high-centrality airports disproportionately originate
maintenance-caused delays?**

A positive correlation would mean operational disruptions cascade from hub airports
— useful for prioritizing ground support and spare parts positioning.

In [ ]:
# Read airport centrality scores into Databricks
centrality_df = spark_query("""
    MATCH (ap:Airport)
    WHERE ap.pagerank_score IS NOT NULL
    RETURN ap.iata               AS iata,
           ap.city               AS city,
           ap.pagerank_score     AS pagerank_score,
           ap.betweenness_score  AS betweenness_score
""")

centrality_df.createOrReplaceTempView("airport_centrality")
display(centrality_df.orderBy("pagerank_score", ascending=False))

In [ ]:
# Read flight delay data grouped by departure airport
delay_by_airport = spark_query("""
    MATCH (f:Flight)-[:DEPARTS_FROM]->(ap:Airport)
    OPTIONAL MATCH (f)-[:HAS_DELAY]->(d:Delay)
    RETURN ap.iata                                                          AS iata,
           count(DISTINCT f)                                                AS total_flights,
           count(d)                                                         AS total_delays,
           count(CASE WHEN d.cause = 'Maintenance' THEN 1 END)              AS maintenance_delays,
           round(avg(CASE WHEN d.cause = 'Maintenance' THEN d.minutes END), 1) AS avg_maint_delay_mins
""")

delay_by_airport.createOrReplaceTempView("airport_delays")
display(delay_by_airport)

In [ ]:
# Join centrality with delay data
correlation = spark.sql("""
    SELECT
        ac.iata,
        ac.city,
        round(ac.pagerank_score, 4)              AS pagerank,
        round(ac.betweenness_score, 1)           AS betweenness,
        ad.total_flights,
        ad.maintenance_delays,
        round(
            100.0 * ad.maintenance_delays / NULLIF(ad.total_flights, 0), 1
        )                                        AS maint_delay_pct,
        ad.avg_maint_delay_mins
    FROM airport_centrality ac
    JOIN airport_delays ad ON ac.iata = ad.iata
    ORDER BY pagerank DESC
""")

display(correlation)

In [ ]:
from pyspark.sql.functions import corr

# Pearson correlation: PageRank vs maintenance delay percentage
pearson = correlation.select(
    corr("pagerank", "maint_delay_pct").alias("pearson_r")
).collect()[0]["pearson_r"]

print(f"Pearson correlation — PageRank vs. Maintenance Delay %: {round(pearson, 4)}")
print()
print("Interpretation:")
if pearson and abs(pearson) > 0.5:
    direction = "positive" if pearson > 0 else "negative"
    print(f"  Strong {direction} correlation: high-centrality airports have")
    print(f"  {'more' if pearson > 0 else 'fewer'} maintenance-caused delay exposure.")
elif pearson and abs(pearson) > 0.2:
    print("  Moderate correlation — centrality is a partial predictor of maintenance delay exposure.")
else:
    print("  Weak correlation — maintenance delays are distributed independently of network centrality.")

## Section 8: Clean Up

In [ ]:
run_query("CALL gds.graph.drop('airport-routes', false) YIELD graphName")
driver.close()
print("Projection dropped. pagerank_score and betweenness_score remain on Airport nodes.")

## Neo4j Browser Queries

**Airports ranked by PageRank:**
```cypher
MATCH (ap:Airport)
WHERE ap.pagerank_score IS NOT NULL
RETURN ap.iata, ap.city, round(ap.pagerank_score, 4) AS pagerank
ORDER BY pagerank DESC
```

**Airports where PageRank and Betweenness disagree most:**
```cypher
MATCH (ap:Airport)
WHERE ap.pagerank_score IS NOT NULL
WITH ap,
     gds.util.nodeProperty(ap, 'pagerank_score') AS pr,
     gds.util.nodeProperty(ap, 'betweenness_score') AS bt
RETURN ap.iata, ap.city,
       round(ap.pagerank_score, 4) AS pagerank,
       round(ap.betweenness_score, 1) AS betweenness
ORDER BY ap.iata
```

**Flights with maintenance delays departing from the top PageRank airport:**
```cypher
MATCH (ap:Airport)<-[:DEPARTS_FROM]-(f:Flight)-[:HAS_DELAY]->(d:Delay {cause: 'Maintenance'})
WITH ap ORDER BY ap.pagerank_score DESC LIMIT 1
MATCH (ap)<-[:DEPARTS_FROM]-(f:Flight)-[:HAS_DELAY]->(d:Delay {cause: 'Maintenance'})
RETURN ap.iata AS airport, f.flight_number, d.minutes AS delay_minutes
ORDER BY d.minutes DESC
```